In [3]:

import numpy as np
import pandas as pd

from scripts.beamforming import matrix_filter
from scripts.data_filter import filter_dataframe
from scripts.data_loader import load_dataframe
from scripts.utils import NETWORK_TYPE, RF_PARAM_5G, get_config

filename = "5G_data_2023.mat"

# Series of random seeds for reproducability
random_seeds = np.loadtxt('../data/random_seeds.csv', dtype=int)

# load the dataframe from saved file or 'raw' matlab file
df = load_dataframe(filename, NETWORK_TYPE._5G)

# # Drop unused columns to save space
matrix_cols_to_drop = ['toa_pps', 'toa_cir', 'toa_cov', 'campaign_id']
df['measurements_matrix'] = df['measurements_matrix'].apply(
    lambda x: x.drop(columns=matrix_cols_to_drop)
)

# base params
rf_param = RF_PARAM_5G.RSRQ
bands = [78]
n_best_beams = 1
n_best_pcis = None

# Get arfcn's in the n78 band
band_config = get_config("band_map.json")
selected_arfcns = [
    int(arfcn)
    for mapping in band_config.values()
    for arfcn, band in mapping.items()
    if band in bands
]

# filter bands and campaigns
df = filter_dataframe(
    df=df,
    operators=None,
    include_columns=[
        "pci",
        "beam_index",
        "nr_arfcn",
        "operator_id",
        rf_param.value,
    ],
    freqs=selected_arfcns,
    campaigns=list(range(0, 21)),

)

# filter n best PCIs and beams
df.loc[:, "measurements_matrix"] = df.loc[:, "measurements_matrix"].apply(
    lambda x: matrix_filter(
        x,
        rf_param,
        include_n_best_pcis=n_best_pcis,
        include_n_best_beams=n_best_beams,
    )
)

print(f"Loaded df of size {df.shape}")

Loaded dataframe from .h5 file: /Users/andreres/Documents/UIO/master/thesis/dev/5G_localization/data/dataframe_cache/5G_data_2023.h5


/Users/andreres/Documents/UIO/master/thesis/dev/5G_localization/scripts/data_filter.py:36: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["measurements_matrix"] = df["measurements_matrix"].apply(


Loaded df of size (8412, 4)


In [4]:
from scripts.utils import dataset_tp_rp_split
from scripts.localization_model import LocalizationModel

all_errors = []
all_stats = []

n_runs = 20
for i in range(n_runs):
    df_tp, df_rp = dataset_tp_rp_split(df, 0.3, random_seeds[i])
    loc_model = LocalizationModel(
        n_clusters=10,
    )
    loc_model.fit(df_rp)
    est_positions = loc_model.predict(df_tp)
    error, stats = loc_model.get_performance_stats(df_tp, est_positions, print_stats=False)
    print(f'Run #{i + 1}: {error.mean():.2f}')
    all_errors.extend(error.tolist())
    all_stats.append(stats)

all_errors = np.array(all_errors)

stats_df = pd.DataFrame(all_stats)

print(f'Number of RPs = {df_rp.shape[0]}')
print(f'Number of TPs = {df_tp.shape[0]}')

print(f"""
====== RESULTS ======
TOTAL MEASUREMENTS {len(all_errors)}
RUNS {n_runs}
MEAN {all_errors.mean():.4f}
STD.DEV {all_errors.std():.4f}
""")

stats_df.to_csv('summary_3.csv')

stats_df.mean()

Run #1: 7.92
Run #2: 7.58
Run #3: 7.63
Run #4: 7.67
Run #5: 7.84
Run #6: 7.78
Run #7: 7.80
Run #8: 7.58
Run #9: 8.14
Run #10: 7.80
Run #11: 7.52
Run #12: 7.89
Run #13: 7.97
Run #14: 8.11
Run #15: 8.24
Run #16: 7.67
Run #17: 7.97
Run #18: 7.79
Run #19: 7.81
Run #20: 7.54
Number of RPs = 5850
Number of TPs = 2562

====== RESULTS ======
TOTAL MEASUREMENTS 50717
RUNS 20
MEAN 7.8132
STD.DEV 10.7747



median_error        3.724689
mean_error          7.812493
std_error          10.757538
min_error           0.000790
max_error         118.934287
training_time       3.164636
inference_time      5.023464
dtype: float64